# Homework 2 - Regression


In [1]:
import numpy as np
import pandas as pd


In [2]:
url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'
df = pd.read_csv(url)


In [3]:
cols = [
    'engine_displacement',
    'horsepower',
    'vehicle_weight',
    'model_year',
    'fuel_efficiency_mpg'
]

df = df[cols]
df.head()


,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
0,2180,243.0,3870,2006,31.9
1,2390,272.0,4210,2008,31.3
2,2320,267.0,4240,1996,27.5
3,2130,258.0,4490,1989,28.5
4,2580,304.0,4510,1994,31.0


### EDA

Look at the `fuel_efficiency_mpg` variable. Does it have a long tail?


In [4]:
df['fuel_efficiency_mpg'].describe()


count    10000.000000
mean        29.997700
std          2.930245
min         19.800000
25%         28.000000
50%         30.000000
75%         31.900000
max         41.200000
Name: fuel_efficiency_mpg, dtype: float64

The variable `fuel_efficiency_mpg` does not have a long tail.


## Question 1

There's one column with missing values. What is it?


In [5]:
df.isnull().sum()


engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

## Question 2

What's the median (50% percentile) for variable `'horsepower'`?


In [6]:
df['horsepower'].median()


np.float64(254.0)

### Prepare and split the dataset


In [7]:
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv.dot(X.T).dot(y)

    return w[0], w[1:]

def train_linear_regression_reg(X, y, r=0.0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    reg = r * np.eye(XTX.shape[0])
    XTX = XTX + reg

    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv.dot(X.T).dot(y)

    return w[0], w[1:]

def rmse(y, y_pred):
    error = y_pred - y
    mse = (error ** 2).mean()
    return np.sqrt(mse)


In [8]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_shuffled = df.iloc[idx].reset_index(drop=True)

df_train = df_shuffled.iloc[:n_train].copy()
df_val = df_shuffled.iloc[n_train:n_train + n_val].copy()
df_test = df_shuffled.iloc[n_train + n_val:].copy()

y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values
y_test = df_test['fuel_efficiency_mpg'].values

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

features = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']


## Question 3

Fill with 0 vs fill with mean (from train only). Which option gives better RMSE?


In [9]:
# Fill with 0
X_train_0 = df_train[features].fillna(0).values
X_val_0 = df_val[features].fillna(0).values

w0_0, w_0 = train_linear_regression(X_train_0, y_train)
y_pred_0 = w0_0 + X_val_0.dot(w_0)
rmse_0 = round(rmse(y_val, y_pred_0), 3)

# Fill with mean
mean_hp = df_train['horsepower'].mean()
X_train_mean = df_train[features].fillna(mean_hp).values
X_val_mean = df_val[features].fillna(mean_hp).values

w0_mean, w_mean = train_linear_regression(X_train_mean, y_train)
y_pred_mean = w0_mean + X_val_mean.dot(w_mean)
rmse_mean = round(rmse(y_val, y_pred_mean), 3)

print(f"fillna(0) RMSE:    {rmse_0}")
print(f"fillna(mean) RMSE: {rmse_mean}")
print(f"Better option:     {'With mean' if rmse_mean < rmse_0 else 'With 0'}")


fillna(0) RMSE:    2.205
fillna(mean) RMSE: 2.202
Better option:     With mean


## Question 4

Regularization with r in [0, 0.01, 0.1, 1, 5, 10, 100]. Which r gives the best RMSE?


In [10]:
r_values = [0, 0.01, 0.1, 1, 5, 10, 100]

for r in r_values:
    w0, w = train_linear_regression_reg(X_train_0, y_train, r=r)
    y_pred = w0 + X_val_0.dot(w)
    score = round(rmse(y_val, y_pred), 4)
    print(f"r = {r:6}: RMSE = {score}")


r =      0: RMSE = 2.2053
r =   0.01: RMSE = 2.2058
r =    0.1: RMSE = 2.2241
r =      1: RMSE = 2.3492
r =      5: RMSE = 2.4094
r =     10: RMSE = 2.4195
r =    100: RMSE = 2.4292


## Question 5

Seed stability: try seeds [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]. What's the standard deviation of scores?


In [11]:
scores = []

for seed in range(10):
    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_shuffled = df.iloc[idx].reset_index(drop=True)

    tr = df_shuffled.iloc[:n_train]
    val = df_shuffled.iloc[n_train:n_train + n_val]

    y_tr = tr['fuel_efficiency_mpg'].values
    y_v = val['fuel_efficiency_mpg'].values

    X_tr = tr[features].fillna(0).values
    X_v = val[features].fillna(0).values

    w0, w = train_linear_regression(X_tr, y_tr)
    y_pred = w0 + X_v.dot(w)
    score = rmse(y_v, y_pred)
    scores.append(score)
    print(f"seed {seed}: RMSE = {score:.4f}")

std_score = round(np.std(scores), 3)
print(f"std: {std_score}")


seed 0: RMSE = 2.2392
seed 1: RMSE = 2.2021
seed 2: RMSE = 2.1634
seed 3: RMSE = 2.2044
seed 4: RMSE = 2.2019
seed 5: RMSE = 2.2458
seed 6: RMSE = 2.2697
seed 7: RMSE = 2.1908
seed 8: RMSE = 2.2166
seed 9: RMSE = 2.2070
std: 0.029


## Question 6

Seed 9, combine train and validation, r=0.001. What's the RMSE on test?


In [12]:
np.random.seed(9)
idx = np.arange(n)
np.random.shuffle(idx)

df_shuffled = df.iloc[idx].reset_index(drop=True)

df_train_9 = df_shuffled.iloc[:n_train]
df_val_9 = df_shuffled.iloc[n_train:n_train + n_val]
df_test_9 = df_shuffled.iloc[n_train + n_val:]

df_full_train = pd.concat([df_train_9, df_val_9]).reset_index(drop=True)

y_full_train = df_full_train['fuel_efficiency_mpg'].values
y_test_9 = df_test_9['fuel_efficiency_mpg'].values

X_full_train = df_full_train[features].fillna(0).values
X_test_9 = df_test_9[features].fillna(0).values

w0, w = train_linear_regression_reg(X_full_train, y_full_train, r=0.001)
y_pred_test = w0 + X_test_9.dot(w)
test_rmse = round(rmse(y_test_9, y_pred_test), 3)

print(f"Test RMSE: {test_rmse}")


Test RMSE: 2.236
